# Regularization in PyTorch: fighting overfitting

_PyTorch_

**Dropout, batch norm, weight decay, early stopping, and grad clipping — the tools that keep a PyTorch model from memorizing the training set.**

Overfitting happens when a model has enough capacity to memorize the training set — it fits the noise, not just the signal. Every technique here is a different way to limit effective capacity or add training variety so memorization is harder than learning the real pattern:

       
         
- Dropout and batch norm add noise / constraints inside the network.
         
- Weight decay shrinks the weights, keeping the function smooth.
         
- Early stopping limits how long the model trains, so it never reaches the deeply-memorized state.
         
- Augmentation grows the apparent dataset so there is simply more to fit.
       

---

This notebook is a practice scaffold for the **Regularization in PyTorch: fighting overfitting** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Build a dataset and model that are easy to overfit

To *see* regularization work, we need a setup that overfits without it: very few training rows and many pure-noise features, where only two features actually carry signal. The model itself bakes in two regularizers — `BatchNorm1d` and `Dropout` — both of which behave differently in train vs eval mode, a distinction that matters later.

In [ ]:
import copy
import torch
import torch.nn as nn

torch.manual_seed(0)

# --- tiny synthetic problem: few samples, many noise features (easy to overfit) ---
N_TRAIN = 64
N_VAL = 400
D = 200

def make(n):
    X = torch.randn(n, D)
    logits = 1.5 * X[:, 0] - 1.2 * X[:, 1]        # only 2 features matter; rest is noise
    y = (torch.rand(n) < torch.sigmoid(logits)).long()
    return X, y

Xtr, ytr = make(N_TRAIN)
Xva, yva = make(N_VAL)

# --- a model WITH batch norm + dropout (regularizers live INSIDE the model) ---
model = nn.Sequential(
    nn.Linear(D, 64),
    nn.BatchNorm1d(64),     # train: batch stats; eval: running stats  (mode-dependent!)
    nn.ReLU(),
    nn.Dropout(0.5),        # train: drops 50% of units; eval: pass-through (mode-dependent!)
    nn.Linear(64, 2),       # 2 logits; CrossEntropyLoss wants RAW logits + class indices
)

loss_fn = nn.CrossEntropyLoss()
# AdamW = decoupled weight decay = the CORRECT way to do L2 with Adam.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

### Step 2 — Train with gradient clipping and early stopping

Each epoch we switch to `model.train()` (dropout on, batch norm using batch stats), take a step, and clip the gradient norm to keep any single update from blowing up. Then we switch to `model.eval()` and measure validation loss under `no_grad`. Early stopping watches that validation loss: whenever it hits a new low we snapshot the weights, and if it fails to improve for `patience` epochs we stop.

In [ ]:
# --- training loop with EARLY STOPPING + GRADIENT CLIPPING ---
best_val = float("inf")
best_state = copy.deepcopy(model.state_dict())   # snapshot of the BEST weights
patience = 15
since_best = 0

for epoch in range(200):
    # ---- train ----
    model.train()                                 # dropout ON, batchnorm uses batch stats
    optimizer.zero_grad()                         # grads accumulate -> always zero first
    logits = model(Xtr)
    loss = loss_fn(logits, ytr)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # cap gradient size
    optimizer.step()

    # ---- validate ----
    model.eval()                                  # dropout OFF, batchnorm uses running stats
    with torch.no_grad():                         # no graph needed at eval -> save memory
        val_loss = loss_fn(model(Xva), yva).item()

    # ---- early stopping: keep the BEST weights, not the last ----
    if val_loss < best_val - 1e-4:
        best_val = val_loss
        best_state = copy.deepcopy(model.state_dict())   # new best -> snapshot it
        since_best = 0
    else:
        since_best += 1
        if since_best >= patience:
            print(f"early stop at epoch {epoch} (best val loss {best_val:.4f})")
            break

### Step 3 — Restore the best weights and report accuracy

The weights at the moment we stop are *not* the best ones — validation loss was already rising. So we reload the snapshot we took at the lowest validation loss, put the model back in `eval` mode, and report final accuracy on the held-out set.

In [ ]:
model.load_state_dict(best_state)                 # RESTORE the best weights before using
model.eval()
with torch.no_grad():
    val_acc = (model(Xva).argmax(1) == yva).float().mean().item()
print(f"restored best val loss = {best_val:.4f} | val accuracy = {val_acc:.3f}")

## Visualize the data & results

_Does regularization actually stop overfitting? Train the same tiny logistic model on few samples with many noise features, WITHOUT vs WITH L2 (weight decay), and plot validation loss over epochs._

### Step 1 — Build a from-scratch overfitting setup

To isolate weight decay we drop PyTorch and hand-write logistic regression in NumPy. The data is deliberately overfit-prone: very few training rows, mostly noise features. We define plain `sigmoid` and binary-cross-entropy helpers, plus a `train` routine whose gradient includes the `+ l2 * w` term — that term *is* weight decay.

In [ ]:
import numpy as np
rng = np.random.RandomState(0)

# tiny REAL overfitting setup: very few samples, mostly noise features
n_train = 30
n_val = 300
d = 400

def make(n):
    X = rng.randn(n, d)
    logits = 1.5*X[:,0] - 1.2*X[:,1]      # only 2 informative features; rest is noise
    y = (rng.rand(n) < 1/(1+np.exp(-logits))).astype(float)
    return X, y

Xtr, ytr = make(n_train)
Xva, yva = make(n_val)

def sigmoid(z):
    return 1/(1+np.exp(-np.clip(z, -30, 30)))

def bce(p, y):
    p = np.clip(p, 1e-9, 1-1e-9)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

def train(l2, epochs=300, lr=0.5):
    w = np.zeros(d)
    b = 0.0
    val = []
    for _ in range(epochs):
        p = sigmoid(Xtr @ w + b)
        g = p - ytr
        gw = Xtr.T @ g / n_train + l2 * w     # the +l2*w term IS weight decay / L2
        gb = g.mean()
        w -= lr*gw
        b -= lr*gb
        val.append(bce(sigmoid(Xva @ w + b), yva))
    return val

### Step 2 — Compare validation loss with and without L2

Now we train twice on the same data: once with no regularization and once with L2 weight decay. Without it, validation loss bottoms out and then climbs as the model memorizes noise; with it, validation loss stays low. The printed start-to-final numbers make the difference concrete.

In [ ]:
val_no = train(l2=0.0)     # no regularization -> overfits, val loss rises
val_l2 = train(l2=0.1)     # L2 weight decay   -> val loss stays low

idx = range(0, 300, 5)     # subsample to 60 points
print("no_reg:", [round(val_no[i], 3) for i in idx])
print("l2    :", [round(val_l2[i], 3) for i in idx])
print("no_reg start->final:", round(val_no[0],3), "->", round(val_no[-1],3))   # 0.812 -> 1.117
print("l2     start->final:", round(val_l2[0],3), "->", round(val_l2[-1],3))    # 0.812 -> 0.836

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Create drop = nn.Dropout(p=0.5) and a fixed input x = torch.ones(1, 10). With drop.train(), run the dropout twice and print both outputs; then call drop.eval() and run it once more. Set torch.manual_seed(0) first.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Toggle the layer mode with drop.train() then drop.eval() around the calls. — _nn.Dropout is mode-dependent: it only zeros (and rescales by $\tfrac{1}{1-p}$) in train mode, and is a pass-through in eval._
- Print the two train-mode outputs and the eval-mode output. — _The two train outputs differ (random masks); the eval output is the unchanged input — the famous train/eval trap made visible._

**Answer:** import torch
import torch.nn as nn
torch.manual_seed(0)
drop = nn.Dropout(p=0.5)
x = torch.ones(1, 10)

drop.train()
print(drop(x))   # some entries 0, survivors scaled to 2.0 (1/(1-0.5))
print(drop(x))   # a DIFFERENT random mask -> different output
drop.eval()
print(drop(x))   # tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]) -- pass-through
# Exact survivors vary, but eval ALWAYS returns x unchanged.

</details>

**Problem 2.** Type this in Colab. Build model = nn.Sequential(nn.Linear(8, 16), nn.Dropout(0.5), nn.Linear(16, 2)). With a fixed input x = torch.randn(4, 8) (seed 0), call model.eval() and run it twice; then call model.train() and run it twice. Print whether the two outputs are equal in each mode with torch.equal.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare repeated forward passes with torch.equal(a, b) in each mode. — _It returns a single boolean, making the determinism difference unmistakable._
- Predict before running: equal in eval(), unequal in train(). — _Dropout is deterministic (identity) in eval but random in train — the #1 source of "my predictions keep changing" bugs._

**Answer:** torch.manual_seed(0)
model = nn.Sequential(nn.Linear(8, 16), nn.Dropout(0.5), nn.Linear(16, 2))
x = torch.randn(4, 8)

model.eval()
print(torch.equal(model(x), model(x)))   # True  -- deterministic in eval
model.train()
print(torch.equal(model(x), model(x)))   # False -- dropout randomizes in train

</details>

**Problem 3.** Type this in Colab. Make bn = nn.BatchNorm1d(3). Push a batch x = torch.randn(16, 3) (seed 0) through it in train() mode, then print bn.running_mean. Predict: is it still all zeros after one training forward pass?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call bn.train() then bn(x), and read bn.running_mean. — _In train mode BatchNorm updates its running statistics from the batch; those running stats are what eval() later uses._
- Predict before running: the running mean moves OFF zero. — _It starts at zeros and is updated toward the batch mean by the momentum (default 0.1), proving train-mode side effects._

**Answer:** torch.manual_seed(0)
bn = nn.BatchNorm1d(3)
print(bn.running_mean)          # tensor([0., 0., 0.]) before
bn.train()
_ = bn(x := torch.randn(16, 3))
print(bn.running_mean)          # small NON-zero values, e.g. tensor([0.0?, ...])
# Updated as 0.9*old + 0.1*batch_mean -> no longer all zeros.

</details>

**Problem 4.** Type this in Colab. Try nn.BatchNorm1d(3)(torch.randn(1, 3)) in train() mode (batch of ONE) and read the error. Then show it works with a batch of 16. Explain in a comment why batch size 1 fails.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run a forward pass with a single sample in train mode and catch the ValueError. — _BatchNorm needs more than one sample to estimate a per-feature variance across the batch._
- Repeat with torch.randn(16, 3). — _A real batch gives a valid variance, so the layer runs — the standard fix is a bigger batch (or GroupNorm/LayerNorm)._

**Answer:** bn = nn.BatchNorm1d(3); bn.train()
try:
    bn(torch.randn(1, 3))
except ValueError as e:
    print("CAUGHT:", str(e).splitlines()[0])
    # Expected more than 1 value per channel when training, got input size torch.Size([1, 3])
print(bn(torch.randn(16, 3)).shape)   # torch.Size([16, 3]) -- fine with a real batch
# Batch size 1 -> variance over the batch is undefined.

</details>

**Problem 5.** Type this in Colab. Create one nn.Linear(4, 1) and two optimizers on its parameters: torch.optim.SGD(p, lr=0.1, weight_decay=0.0) and torch.optim.SGD(p, lr=0.1, weight_decay=10.0). Take one step on each (with the SAME gradient) and print the weight norm. Predict which optimizer shrinks the weights more.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Backprop a loss, then call step() for each optimizer on a fresh copy of the layer. — _weight_decay adds $-\eta\lambda w$ to the update, so larger decay pulls weights harder toward zero._
- Compare weight.norm() before and after each step. — _Predict: weight_decay=10.0 produces a smaller post-step norm — that shrink IS L2 regularization._

**Answer:** import copy
torch.manual_seed(0)
base = nn.Linear(4, 1)
x, y = torch.randn(8, 4), torch.randn(8, 1)

def step(wd):
    m = copy.deepcopy(base)
    opt = torch.optim.SGD(m.parameters(), lr=0.1, weight_decay=wd)
    opt.zero_grad()
    ((m(x) - y) ** 2).mean().backward()
    opt.step()
    return m.weight.norm().item()

print(round(step(0.0), 4))    # e.g. 0.7261  (no decay)
print(round(step(10.0), 4))   # SMALLER     (heavy decay shrinks weights more)

</details>

**Problem 6.** Type this in Colab. Create gradients by backpropagating a loss through nn.Linear(4, 1). Compute the total gradient norm; then call nn.utils.clip_grad_norm_(params, max_norm=0.5) and recompute it. Print both. (Use seed 0.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Sum per-parameter grad.norm()**2 and square-root for the total norm. — _clip_grad_norm_ rescales all gradients so this combined L2 norm is at most max_norm._
- Call clip_grad_norm_ between backward() and step() and recompute the norm. — _If the original norm exceeded 0.5, the clipped norm is exactly 0.5 — preventing a single huge update._

**Answer:** torch.manual_seed(0)
m = nn.Linear(4, 1)
x, y = torch.randn(8, 4), torch.randn(8, 1)
((m(x) - y) ** 2).mean().backward()

def total_norm():
    return sum(p.grad.norm()**2 for p in m.parameters()) ** 0.5
print(round(float(total_norm()), 4))                       # original (e.g. > 0.5)
nn.utils.clip_grad_norm_(m.parameters(), max_norm=0.5)
print(round(float(total_norm()), 4))                       # 0.5 (clipped to the cap)

</details>

**Problem 7.** Type this in Colab. Simulate early stopping by hand: given the list val = [0.9, 0.7, 0.55, 0.6, 0.62, 0.65], track the best (lowest) value and the epoch it occurred, stopping once validation has not improved for patience=2 epochs. Print the best value and best epoch.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Keep best_val and since_best counters, snapshotting on each new low. — _In real training you would copy.deepcopy(model.state_dict()) at each best — here we track the index instead._
- Break when since_best >= patience. — _You must restore the BEST weights, not the last ones, which are worse after validation started rising._

**Answer:** val = [0.9, 0.7, 0.55, 0.6, 0.62, 0.65]
best_val, best_epoch, since_best, patience = float("inf"), -1, 0, 2
for epoch, v in enumerate(val):
    if v = patience:
            print(f"early stop at epoch {epoch}")   # early stop at epoch 4
            break
print("best:", best_val, "at epoch", best_epoch)     # best: 0.55 at epoch 2

</details>

**Problem 8.** Type this in Colab. Build a small regularized block nn.Sequential(nn.Linear(20, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 2)) and an AdamW optimizer with weight_decay=1e-2. Run ONE train step on x = torch.randn(16, 20), y = torch.randint(0, 2, (16,)) (seed 0), printing the loss; then switch to eval() and print the prediction shape under torch.no_grad().

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use model.train() + zero_grad() + backward() + step() for the train step, with nn.CrossEntropyLoss. — _CrossEntropy wants raw logits and long class indices; AdamW applies decoupled (correct) weight decay._
- Call model.eval() and wrap inference in torch.no_grad(). — _eval turns Dropout into a pass-through and BatchNorm onto running stats; no_grad skips graph bookkeeping at inference._

**Answer:** torch.manual_seed(0)
model = nn.Sequential(nn.Linear(20, 32), nn.BatchNorm1d(32),
                      nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 2))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
x = torch.randn(16, 20)
y = torch.randint(0, 2, (16,))

model.train()
opt.zero_grad()
loss = nn.CrossEntropyLoss()(model(x), y)
loss.backward(); opt.step()
print(round(loss.item(), 4))            # a finite loss near ln(2) ~= 0.69

model.eval()
with torch.no_grad():
    print(model(x).shape)               # torch.Size([16, 2])

</details>